In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA RTX 5000 Ada Generation


# For FWI dataset

In [ ]:
pip install scikit-image

  Using cached scikit_image-0.26.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (15 kB)
  Using cached tifffile-2026.3.3-py3-none-any.whl.metadata (31 kB)
Using cached scikit_image-0.26.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (13.6 MB)
Using cached tifffile-2026.3.3-py3-none-any.whl (243 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-image] [scikit-image]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
 pip install POT

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 5.4 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


# WFM trained separately for each dataset

# With more complex unet

In [ ]:
"""
WFM (Wasserstein Flow Matching) — Single Family Training
Trains one model per family, saves best, evaluates on val set.
Families : CurveVelA | FlatVelA | CurveVelB
"""

import os, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from skimage.metrics import structural_similarity as skssim

try:
    import ot
except ImportError:
    raise ImportError("pip install POT")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ═══════════════════════════════════════════════════════════════════════
# 1.  DATASET
# ═══════════════════════════════════════════════════════════════════════

FAMILY_DIRS = {
    "CurveVelA" : ("CurveVel_A", "data", "model"),
    "FlatVelA"  : ("FlatVel_A",  "data", "model"),
    "CurveVelB" : ("CurveVel_B", "data", "model"),
}

GLOBAL_VMIN = 1500.0
GLOBAL_VMAX = 4500.0


def load_family_arrays(base_dir, family_name):
    folder, data_sub, model_sub = FAMILY_DIRS[family_name]
    data_dir  = os.path.join(base_dir, folder, data_sub)
    model_dir = os.path.join(base_dir, folder, model_sub)

    data_files  = sorted(glob.glob(os.path.join(data_dir,  "data*.npy")))
    model_files = sorted(glob.glob(os.path.join(model_dir, "model*.npy")))

    assert len(data_files) > 0, f"No files in {data_dir}"
    assert len(data_files) == len(model_files)

    X = np.concatenate([np.load(f) for f in data_files],  axis=0)
    Y = np.concatenate([np.load(f) for f in model_files], axis=0)
    if Y.ndim == 3:
        Y = Y[:, None]
    return X.astype(np.float32), Y.astype(np.float32)


class SingleFamilyDataset(Dataset):
    def __init__(self, X, Y, vmin=GLOBAL_VMIN, vmax=GLOBAL_VMAX):
        self.X    = X
        self.Y    = Y
        self.vmin = vmin
        self.vmax = vmax

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        seis = torch.from_numpy(self.X[idx].copy())
        vel  = torch.from_numpy(self.Y[idx].copy())
        vel  = (vel  - self.vmin) / (self.vmax - self.vmin + 1e-8)
        seis = (seis - seis.mean()) / (seis.std() + 1e-8)
        return vel, seis


def build_dataloaders(base_dir, family_name,
                      batch_size=48, val_ratio=0.2, seed=42):
    X, Y = load_family_arrays(base_dir, family_name)
    print(f"  {family_name}  N={len(X)}  "
          f"vmin={Y.min():.1f}  vmax={Y.max():.1f}")

    ds    = SingleFamilyDataset(X, Y)
    n_val = max(1, int(len(ds) * val_ratio))
    n_tr  = len(ds) - n_val
    gen   = torch.Generator().manual_seed(seed)
    tr, va = random_split(ds, [n_tr, n_val], generator=gen)

    train_loader = DataLoader(tr, batch_size=batch_size,
                              shuffle=True,  drop_last=True)
    val_loader   = DataLoader(va, batch_size=batch_size,
                              shuffle=False)
    return train_loader, val_loader


# ═══════════════════════════════════════════════════════════════════════
# 2.  MODEL
# ═══════════════════════════════════════════════════════════════════════

class SeismicEncoder(nn.Module):
    """Compresses (5, 1000, 70) seismic input → (B, out_ch, 70, 70)."""
    def __init__(self, out_channels=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(5,   16,  (7,3), stride=(2,1), padding=(3,1)), nn.ReLU(),
            nn.Conv2d(16,  32,  (7,3), stride=(2,1), padding=(3,1)), nn.ReLU(),
            nn.Conv2d(32,  64,  (5,3), stride=(2,1), padding=(2,1)), nn.ReLU(),
            nn.Conv2d(64,  128, (5,3), stride=(2,1), padding=(2,1)), nn.ReLU(),
            nn.Conv2d(128, 128, (5,3), stride=(2,1), padding=(2,1)), nn.ReLU(),
            nn.AdaptiveAvgPool2d((70, 70)),
        )
        self.proj = nn.Conv2d(128, out_channels, 1)

    def forward(self, y):
        return self.proj(self.net(y))   # (B, out_ch, 70, 70)


class TimeEmbed(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1, 64),  nn.SiLU(),
            nn.Linear(64, embed_dim), nn.SiLU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, t):
        return self.mlp(t)


class ConvBlock(nn.Module):
    """GroupNorm + FiLM time conditioning."""
    def __init__(self, in_c, out_c, embed_dim=128):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c,  out_c, 3, padding=1)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1)
        self.norm1 = nn.GroupNorm(min(8, out_c), out_c)
        self.norm2 = nn.GroupNorm(min(8, out_c), out_c)
        self.film  = nn.Linear(embed_dim, out_c * 2)

    def forward(self, x, emb):
        sc, sh = self.film(emb).chunk(2, dim=-1)
        h = F.silu(self.norm1(self.conv1(x)))
        h = h * (1 + sc[:,:,None,None]) + sh[:,:,None,None]
        h = F.silu(self.norm2(self.conv2(h)))
        return h


class UNet(nn.Module):
    """UNet with seismic encoder + skip connections + FiLM conditioning."""
    COND_CH = 32
    EMB_DIM = 128

    def __init__(self):
        super().__init__()
        self.seis_enc = SeismicEncoder(self.COND_CH)
        self.t_embed  = TimeEmbed(self.EMB_DIM)

        ic = 1 + self.COND_CH   # velocity + seismic features

        # Encoder
        self.c1   = ConvBlock(ic,   64,  self.EMB_DIM)
        self.c2   = ConvBlock(64,  128,  self.EMB_DIM)
        self.c3   = ConvBlock(128, 256,  self.EMB_DIM)
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.mid  = ConvBlock(256, 256, self.EMB_DIM)

        # Decoder
        self.up1  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.d1   = ConvBlock(128+256, 128, self.EMB_DIM)
        self.up2  = nn.ConvTranspose2d(128,  64, 2, stride=2)
        self.d2   = ConvBlock( 64+128,  64, self.EMB_DIM)
        self.up3  = nn.ConvTranspose2d( 64,  32, 2, stride=2)
        self.d3   = ConvBlock( 32+ 64,  32, self.EMB_DIM)

        self.out  = nn.Conv2d(32, 1, 1)

    @staticmethod
    def _match(x, ref):
        if x.shape[-2:] != ref.shape[-2:]:
            x = F.interpolate(x, size=ref.shape[-2:],
                              mode='bilinear', align_corners=False)
        return x

    def forward(self, x, t, y):
        emb  = self.t_embed(t)
        cond = self.seis_enc(y)
        inp  = torch.cat([x, cond], 1)

        x1 = self.c1(inp,           emb)
        x2 = self.c2(self.pool(x1), emb)
        x3 = self.c3(self.pool(x2), emb)
        h  = self.mid(self.pool(x3), emb)

        h = self._match(self.up1(h), x3)
        h = self.d1(torch.cat([h, x3], 1), emb)
        h = self._match(self.up2(h), x2)
        h = self.d2(torch.cat([h, x2], 1), emb)
        h = self._match(self.up3(h), x1)
        h = self.d3(torch.cat([h, x1], 1), emb)

        return self.out(h)


# ═══════════════════════════════════════════════════════════════════════
# 3.  WFM LOSS
# ═══════════════════════════════════════════════════════════════════════

def image_gradients(z):
    return z[:,:,1:,:]-z[:,:,:-1,:], z[:,:,:,1:]-z[:,:,:,:-1]


def wfm_loss(model, x1, y, grad_weight=0.1):
    """OT sample pairing + flow matching loss."""
    B  = x1.size(0)
    x0 = torch.randn_like(x1)

    # OT matching in pixel space
    x0_np = x0.view(B,-1).detach().cpu().double().numpy()
    x1_np = x1.view(B,-1).detach().cpu().double().numpy()

    c = ot.dist(x0_np, x1_np, metric='sqeuclidean')
    c = c / (c.max() + 1e-8)
    g = ot.emd(np.ones(B)/B, np.ones(B)/B, c)

    # soft sampling from transport plan
    gt  = torch.tensor(g, dtype=torch.float32, device=x1.device)
    gt  = gt / (gt.sum(1, keepdim=True) + 1e-8)
    idx = torch.multinomial(gt, 1).squeeze(1)

    x1_match = x1[idx]
    y_match  = y[idx]

    # flow matching
    t      = torch.rand(B, 1, device=x1.device)
    xt     = (1-t.view(B,1,1,1))*x0 + t.view(B,1,1,1)*x1_match
    target = x1_match - x0
    pred   = model(xt, t, y_match)

    mse    = ((pred - target)**2).mean()
    ph, pw = image_gradients(pred)
    th, tw = image_gradients(target)
    return mse + grad_weight*(((ph-th)**2).mean() + ((pw-tw)**2).mean())


# ═══════════════════════════════════════════════════════════════════════
# 4.  SAMPLING
# ═══════════════════════════════════════════════════════════════════════

@torch.no_grad()
def sample_velocity(model, y, steps=100):
    model.eval()
    B  = y.size(0)
    x  = torch.randn(B, 1, 70, 70, device=y.device)
    dt = 1.0 / steps
    for i in range(steps):
        t = torch.full((B, 1), i/steps, device=y.device)
        x = x + model(x, t, y) * dt
    return x


@torch.no_grad()
def sample_multiple(model, y, n_samples=10, steps=100):
    """
    n_samples stochastic forward passes from different x0.
    Returns (n_samples, B, 1, H, W).
    """
    model.eval()
    all_preds = []
    for _ in range(n_samples):
        all_preds.append(sample_velocity(model, y, steps=steps))
    return torch.stack(all_preds, dim=0)


# ═══════════════════════════════════════════════════════════════════════
# 5.  METRICS
# ═══════════════════════════════════════════════════════════════════════

def denormalize(x, vmin=GLOBAL_VMIN, vmax=GLOBAL_VMAX):
    return x * (vmax - vmin) + vmin


def compute_metrics(pred, gt):
    dr  = float(GLOBAL_VMAX - GLOBAL_VMIN)
    mse = np.mean((pred - gt)**2)
    return {
        "MAE" : float(np.mean(np.abs(pred - gt))),
        "RMSE": float(np.sqrt(mse)),
        "PSNR": float(10 * np.log10(dr**2 / (mse + 1e-8))),
        "SSIM": float(skssim(gt, pred, data_range=dr)),
        "REL" : float(np.mean(np.abs(pred-gt)) /
                      (gt.max()-gt.min()+1e-8) * 100),
    }


# ═══════════════════════════════════════════════════════════════════════
# 6.  EVALUATE  (defined BEFORE train_one_family so it can be called inside)
# ═══════════════════════════════════════════════════════════════════════

def evaluate(model, val_loader, steps=100):
    """Returns mean metrics over full val set in m/s."""
    model.eval()
    all_metrics = []

    with torch.no_grad():
        for vel, seis in val_loader:
            vel  = vel.to(device)
            seis = seis.to(device)

            pred    = sample_velocity(model, seis, steps=steps)
            pred_np = denormalize(pred.cpu().numpy())
            gt_np   = denormalize(vel.cpu().numpy())

            for i in range(pred_np.shape[0]):
                all_metrics.append(
                    compute_metrics(pred_np[i,0], gt_np[i,0]))

    keys = all_metrics[0].keys()
    return {k: float(np.mean([m[k] for m in all_metrics]))
            for k in keys}


# ═══════════════════════════════════════════════════════════════════════
# 7.  TRAIN ONE FAMILY
# ═══════════════════════════════════════════════════════════════════════

def train_one_family(model, train_loader, val_loader,
                     family_name, epochs=500, save_path=None):

    if save_path is None:
        save_path = f"WFM_{family_name}.pt"

    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=3e-4,
        steps_per_epoch=len(train_loader),
        epochs=epochs, pct_start=0.05,
    )

    best_rmse    = float('inf')
    loss_history = []

    for epoch in range(epochs):
        model.train()
        total = 0.0

        pbar = tqdm(train_loader,
                    desc=f'[{family_name}] Epoch {epoch:4d}/{epochs}',
                    leave=False)
        for vel, seis in pbar:
            vel  = vel.to(device)
            seis = seis.to(device)

            loss = wfm_loss(model, vel, seis, grad_weight=0.1)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sch.step()
            total += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.4f}')

        train_loss = total / len(train_loader)
        loss_history.append(train_loss)

        # validate every 5 epochs with fast steps=20
        if epoch % 5 == 0:
            results  = evaluate(model, val_loader, steps=20)
            val_rmse = results["RMSE"]
            print(f"  [{family_name}] Epoch {epoch:4d} | "
                  f"Loss {train_loss:.4f} | "
                  f"RMSE {val_rmse:.1f} m/s | "
                  f"Best {best_rmse:.1f} m/s")

            if val_rmse < best_rmse:
                best_rmse = val_rmse
                torch.save({
                    'epoch'    : epoch,
                    'model'    : model.state_dict(),
                    'val_rmse' : val_rmse,
                    'family'   : family_name,
                }, save_path)
                print(f"    ✓ Saved  (RMSE={best_rmse:.1f} m/s)")

    return loss_history, best_rmse


# ═══════════════════════════════════════════════════════════════════════
# 8.  VISUALIZATION
# ═══════════════════════════════════════════════════════════════════════

def visualize_family(model, val_loader, family_name,
                     save_dir='.', n_samples=10, steps=100,
                     n_show=2, vmin=GLOBAL_VMIN, vmax=GLOBAL_VMAX):
    """
    For each of n_show val samples:
      Col 1 : Ground Truth
      Col 2 : Posterior Mean   (mean over n_samples stochastic runs)
      Col 3 : Error Map        |mean − GT|
      Col 4 : Uncertainty Map  (std over n_samples stochastic runs)
    """
    model.eval()

    # collect n_show samples
    vel_list, seis_list = [], []
    for vel, seis in val_loader:
        vel_list.append(vel)
        seis_list.append(seis)
        if sum(v.shape[0] for v in vel_list) >= n_show:
            break

    vel_all  = torch.cat(vel_list,  dim=0)[:n_show]   # (n_show, 1, H, W)
    seis_all = torch.cat(seis_list, dim=0)[:n_show]   # (n_show, 5, T, R)
    seis_all = seis_all.to(device)

    # n_samples stochastic predictions → (n_samples, n_show, 1, H, W)
    all_preds = sample_multiple(model, seis_all,
                                n_samples=n_samples, steps=steps)

    gt_ms          = denormalize(vel_all.numpy())           # (n_show, 1, H, W)
    preds_ms       = denormalize(all_preds.cpu().numpy())   # (n_samples, n_show, 1, H, W)
    posterior_mean = preds_ms.mean(axis=0)                  # (n_show, 1, H, W)
    posterior_std  = preds_ms.std(axis=0)                   # (n_show, 1, H, W)
    error_map      = np.abs(posterior_mean - gt_ms)

    col_titles = [
        'Ground Truth',
        'Posterior Mean',
        'Error  |mean − GT|',
        f'Uncertainty  (std, {n_samples} samples)',
    ]

    fig = plt.figure(figsize=(16, 4 * n_show))
    gs  = gridspec.GridSpec(n_show, 4, figure=fig,
                            hspace=0.35, wspace=0.25)

    for row in range(n_show):
        gt   = gt_ms[row, 0]
        mean = posterior_mean[row, 0]
        err  = error_map[row, 0]
        std  = posterior_std[row, 0]

        err_max = max(float(err.max()), 1.0)
        std_max = max(float(std.max()), 1.0)

        panels = [
            (gt,   'seismic', vmin,    vmax,    f'{vmin:.0f} m/s', f'{vmax:.0f} m/s'),
            (mean, 'seismic', vmin,    vmax,    f'{vmin:.0f} m/s', f'{vmax:.0f} m/s'),
            (err,  'hot',     0,       err_max, '0',               f'{err_max:.0f} m/s'),
            (std,  'plasma',  0,       std_max, '0',               f'{std_max:.0f} m/s'),
        ]

        for col, (data, cmap, lo, hi, cbar_lo, cbar_hi) in enumerate(panels):
            ax = fig.add_subplot(gs[row, col])
            im = ax.imshow(data, cmap=cmap, vmin=lo, vmax=hi,
                           aspect='auto', origin='upper')
            cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.ax.tick_params(labelsize=7)
            cbar.set_ticks([lo, hi])
            cbar.set_ticklabels([cbar_lo, cbar_hi])
            ax.set_xticks([]); ax.set_yticks([])

            if row == 0:
                ax.set_title(col_titles[col], fontsize=11, fontweight='bold')
            if col == 0:
                ax.set_ylabel(f'Sample {row+1}', fontsize=9)

            # per-panel stats as xlabel
            if col == 0:
                ax.set_xlabel(
                    f'GT range: [{data.min():.0f}, {data.max():.0f}] m/s',
                    fontsize=7)
            elif col == 1:
                rmse = np.sqrt(np.mean((mean - gt)**2))
                ax.set_xlabel(f'RMSE={rmse:.1f} m/s', fontsize=7)
            elif col == 2:
                ax.set_xlabel(
                    f'MAE={err.mean():.1f}  Max={err.max():.1f} m/s',
                    fontsize=7)
            elif col == 3:
                ax.set_xlabel(
                    f'Mean std={std.mean():.1f}  Max={std.max():.1f} m/s',
                    fontsize=7)

    fig.suptitle(f'WFM Posterior Visualization — {family_name}',
                 fontsize=14, fontweight='bold', y=1.01)

    os.makedirs(save_dir, exist_ok=True)
    out_path = os.path.join(save_dir, f'viz_{family_name}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Visualization saved → {out_path}')
    return out_path


def plot_summary_all_families(families, base_dir, save_dir='./viz_outputs'):
    """
    One combined figure: rows = families, cols = GT / Mean / Error / Uncertainty.
    Uses already-saved best checkpoints.
    """
    fig, axes = plt.subplots(len(families), 4,
                             figsize=(16, 4 * len(families)))
    if len(families) == 1:
        axes = axes[None]

    col_titles = ['Ground Truth', 'Posterior Mean',
                  'Error |mean − GT|', 'Uncertainty (std)']

    for row_i, family in enumerate(families):
        ckpt_path = f'WFM_{family}.pt'
        ckpt      = torch.load(ckpt_path, map_location=device,
                               weights_only=False)
        model_tmp = UNet().to(device)
        model_tmp.load_state_dict(ckpt['model'])
        model_tmp.eval()

        _, val_loader_tmp = build_dataloaders(base_dir, family, batch_size=4)

        vel, seis = next(iter(val_loader_tmp))
        vel  = vel[:1]
        seis = seis[:1].to(device)

        preds    = sample_multiple(model_tmp, seis, n_samples=10, steps=50)
        gt_ms    = denormalize(vel.numpy())[0, 0]
        preds_ms = denormalize(preds.cpu().numpy())[:, 0, 0]
        mean_ms  = preds_ms.mean(0)
        std_ms   = preds_ms.std(0)
        err_ms   = np.abs(mean_ms - gt_ms)

        data_panels = [
            (gt_ms,   'seismic', GLOBAL_VMIN,    GLOBAL_VMAX),
            (mean_ms, 'seismic', GLOBAL_VMIN,    GLOBAL_VMAX),
            (err_ms,  'hot',     0,               max(float(err_ms.max()), 1.0)),
            (std_ms,  'plasma',  0,               max(float(std_ms.max()), 1.0)),
        ]

        for col, (data, cmap, lo, hi) in enumerate(data_panels):
            ax = axes[row_i, col]
            im = ax.imshow(data, cmap=cmap, vmin=lo, vmax=hi,
                           aspect='auto', origin='upper')
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            ax.set_xticks([]); ax.set_yticks([])
            if row_i == 0:
                ax.set_title(col_titles[col], fontsize=10, fontweight='bold')
            if col == 0:
                ax.set_ylabel(family, fontsize=10, fontweight='bold')

    plt.suptitle('WFM — All Families Summary', fontsize=14, fontweight='bold')
    plt.tight_layout()
    os.makedirs(save_dir, exist_ok=True)
    summary_path = os.path.join(save_dir, 'summary_all_families.png')
    plt.savefig(summary_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Summary figure → {summary_path}')


def plot_loss_curves(histories, save_dir='./viz_outputs'):
    """Plot training loss curves for all families on one figure."""
    os.makedirs(save_dir, exist_ok=True)
    plt.figure(figsize=(8, 4))
    for family, history in histories.items():
        plt.plot(history, label=family)
    plt.xlabel('Epoch'); plt.ylabel('WFM Loss')
    plt.title('Training Loss — All Families')
    plt.legend(); plt.tight_layout()
    path = os.path.join(save_dir, 'loss_curves.png')
    plt.savefig(path, dpi=150)
    plt.close()
    print(f'  Loss curves → {path}')


# ═══════════════════════════════════════════════════════════════════════
# 9.  COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════════════

def print_comparison_table(all_results):
    """
    all_results : {'CurveVelA': {MAE:.., RMSE:.., SSIM:..}, ...}
    Baselines from OpenFWI (Deng et al. NeurIPS 2022).
    CurveVelB baselines marked as N/A (not reported in paper).
    """
    baselines = {
        "InversionNet": {
            "CurveVelA" : {"MAE": 205.5, "RMSE": 381.9, "SSIM": 0.8074},
            "FlatVelA"  : {"MAE":  39.3, "RMSE":  63.3, "SSIM": 0.9895},
            "CurveVelB" : {"MAE": float('nan'), "RMSE": float('nan'), "SSIM": float('nan')},
        },
        "VelocityGAN": {
            "CurveVelA" : {"MAE": 144.6, "RMSE": 310.2, "SSIM": 0.8624},
            "FlatVelA"  : {"MAE":  35.4, "RMSE":  53.4, "SSIM": 0.9916},
            "CurveVelB" : {"MAE": float('nan'), "RMSE": float('nan'), "SSIM": float('nan')},
        },
    }

    families = ["CurveVelA", "FlatVelA", "CurveVelB"]

    def fmt(v):
        return f'{v:>9.1f}' if not np.isnan(v) else f'{"N/A":>9}'

    print("\n" + "═"*85)
    print(f"  {'Model':<20}", end="")
    for fam in families:
        print(f"{fam:>27}", end="")
    print()
    print(f"  {'':20}", end="")
    for _ in families:
        print(f"{'MAE':>9}{'RMSE':>9}{'SSIM':>9}", end="")
    print()
    print("─"*85)

    for model_name, fam_res in baselines.items():
        print(f"  {model_name:<20}", end="")
        for fam in families:
            r = fam_res[fam]
            ssim_str = f'{r["SSIM"]:>9.4f}' if not np.isnan(r["SSIM"]) else f'{"N/A":>9}'
            print(f'{fmt(r["MAE"])}{fmt(r["RMSE"])}{ssim_str}', end="")
        print()

    print("─"*85)
    print(f"  {'WFM (ours)':<20}", end="")
    for fam in families:
        r    = all_results.get(fam, {})
        mae  = r.get("MAE",  float('nan'))
        rmse = r.get("RMSE", float('nan'))
        ssim = r.get("SSIM", float('nan'))
        ssim_str = f'{ssim:>9.4f}' if not np.isnan(ssim) else f'{"N/A":>9}'
        print(f'{fmt(mae)}{fmt(rmse)}{ssim_str}', end="")
    print()
    print("═"*85)
    print("  Note: baselines from OpenFWI paper (24K–48K samples).")
    print("        WFM trained on ~1350 samples per family.")
    print("        CurveVelB baselines not reported in original paper.")


# ═══════════════════════════════════════════════════════════════════════
# 10. MAIN
# ═══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    BASE_DIR = "/DATA/Flow_Matching"
    EPOCHS   = 300
    FAMILIES = ["CurveVelA", "FlatVelA", "CurveVelB"]
    VIZ_DIR  = "./viz_outputs"

    all_results   = {}
    loss_histories = {}

    for family in FAMILIES:

        print(f"\n{'='*55}")
        print(f"  Training WFM on  {family}")
        print(f"{'='*55}")

        # data
        train_loader, val_loader = build_dataloaders(
            BASE_DIR, family, batch_size=48)

        # model
        model = UNet().to(device)
        print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

        # train
        save_path = f"WFM_{family}.pt"
        history, best_rmse = train_one_family(
            model, train_loader, val_loader,
            family_name = family,
            epochs      = EPOCHS,
            save_path   = save_path,
        )
        loss_histories[family] = history

        # load best checkpoint
        ckpt = torch.load(save_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model'])
        model.eval()
        print(f"\n  Loaded best epoch {ckpt['epoch']} "
              f"(RMSE={ckpt['val_rmse']:.1f} m/s)")

        # final evaluation with steps=100
        print(f"  Running full val evaluation (steps=100) …")
        results = evaluate(model, val_loader, steps=100)
        all_results[family] = results

        print(f"\n  ── {family} Final Results ──")
        for k, v in results.items():
            print(f"    {k:<6}: {v:.4f}")

        # per-family visualization
        print(f"  Generating visualizations …")
        visualize_family(
            model, val_loader,
            family_name = family,
            save_dir    = VIZ_DIR,
            n_samples   = 10,
            steps       = 100,
            n_show      = 2,
        )

    # comparison table
    print_comparison_table(all_results)

    # loss curves
    plot_loss_curves(loss_histories, save_dir=VIZ_DIR)

    # combined summary figure
    print("\n  Generating combined summary figure …")
    plot_summary_all_families(FAMILIES, BASE_DIR, save_dir=VIZ_DIR)

    print(f"\n  All outputs saved to: {VIZ_DIR}")

Device: cuda

  Training WFM on  CurveVelA
  CurveVelA  N=30000  vmin=1500.0  vmax=4500.0
  Parameters: 3,974,113


  [CurveVelA] Epoch    0 | Loss 0.3042 | RMSE 671.3 m/s | Best inf m/s
    ✓ Saved  (RMSE=671.3 m/s)


  [CurveVelA] Epoch    5 | Loss 0.0473 | RMSE 576.3 m/s | Best 671.3 m/s
    ✓ Saved  (RMSE=576.3 m/s)


  [CurveVelA] Epoch   10 | Loss 0.0288 | RMSE 478.0 m/s | Best 576.3 m/s
    ✓ Saved  (RMSE=478.0 m/s)


  [CurveVelA] Epoch   15 | Loss 0.0215 | RMSE 373.0 m/s | Best 478.0 m/s
    ✓ Saved  (RMSE=373.0 m/s)


...

  [CurveVelB] Epoch  280 | Loss 0.0136 | RMSE 354.6 m/s | Best 354.4 m/s


  [CurveVelB] Epoch  285 | Loss 0.0137 | RMSE 354.5 m/s | Best 354.4 m/s


  [CurveVelB] Epoch  290 | Loss 0.0137 | RMSE 354.9 m/s | Best 354.4 m/s


  [CurveVelB] Epoch  295 | Loss 0.0134 | RMSE 354.1 m/s | Best 354.4 m/s
    ✓ Saved  (RMSE=354.1 m/s)



  Loaded best epoch 295 (RMSE=354.1 m/s)
  Running full val evaluation (steps=100) …

  ── CurveVelB Final Results ──
    MAE   : 191.0700
    RMSE  : 357.0516
    PSNR  : 19.1262
    SSIM  : 0.7259
    REL   : 9.2775
  Generating visualizations …
  Visualization saved → ./viz_outputs/viz_CurveVelB.png

═════════════════════════════════════════════════════════════════════════════════════
  Model                                 CurveVelA                   FlatVelA                  CurveVelB
                            MAE     RMSE     SSIM      MAE     RMSE     SSIM      MAE     RMSE     SSIM
─────────────────────────────────────────────────────────────────────────────────────
  InversionNet            205.5    381.9   0.8074     39.3     63.3   0.9895      N/A      N/A      N/A
  VelocityGAN             144.6    310.2   0.8624     35.4     53.4   0.9916      N/A      N/A      N/A
─────────────────────────────────────────────────────────────────────────────────────
  WFM (ours)        